In [ ]:
# 1. IMPORT ALL LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, Dropout, MaxPooling1D, BatchNormalization
import shap

import warnings
warnings.filterwarnings('ignore')

# 2. LOAD DATA
print("Loading data...")
df = pd.read_csv('diabetes.csv')
X = df.drop('T', axis=1) # Features
y = df['T']              # Target

# 3. THREE-WAY DATA SPLIT (70% Train, 15% Validation, 15% Test)
print("Splitting data into 70% Train, 15% Val, 15% Test...")
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
val_fraction = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_fraction, random_state=42, stratify=y_temp)

# 4. STANDARDIZE FEATURES (Fit ONLY on Train data to prevent data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. RESHAPE FOR 1D-CNN [Samples, TimeSteps/Features, Channels]
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_val_cnn = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# 6. BUILD 1D-CNN ARCHITECTURE
print("Building 1D-CNN model...")
model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

# 7. TRAIN THE MODEL
print("Training model... (this may take a few moments)")
history = model.fit(X_train_cnn, y_train, 
                    epochs=100, 
                    batch_size=32, 
                    validation_data=(X_val_cnn, y_val), 
                    verbose=0) # Set to 1 if you want to see epoch progress

# 8. EVALUATE ON UNSEEN TEST DATA
print("\n--- FINAL MODEL EVALUATION (ON 15% TEST SET) ---")
y_pred_prob = model.predict(X_test_cnn, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_pred_prob):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

# Generate and Plot the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['No DR (0)', 'DR (1)'], 
            yticklabels=['No DR (0)', 'DR (1)'])
plt.title('Confusion Matrix - 1D-CNN')
plt.ylabel('True Clinical Diagnosis')
plt.xlabel('Algorithm Prediction')
plt.tight_layout()
plt.show()

# 9. PLOT TRAINING HISTORY (Proof of no overfitting)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

# 10. EXPLAINABILITY WITH SHAP 
print("\nGenerating SHAP Explainability Plot...")

# Wrapper function for the CNN
def cnn_predict(data_2d):
    data_3d = data_2d.reshape(-1, X.shape[1], 1)
    return model.predict(data_3d, verbose=0).flatten()


background_summary = shap.kmeans(X_train_scaled, 10)

# Initialize KernelExplainer 
explainer = shap.KernelExplainer(cnn_predict, background_summary)


print("Calculating SHAP values (Optimized - should be much faster)...")
shap_values = explainer.shap_values(X_test_scaled[:20], silent=True)

# Plot the summary
plt.figure(figsize=(8, 5))
plt.title("SHAP Feature Importance (1D-CNN)")
shap.summary_plot(shap_values, X_test_scaled[:20], feature_names=X.columns)